In [1]:
def file_a_stringa(percorso_file):
    try:
        with open(percorso_file, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        return "Errore: Il file non è stato trovato."
    except Exception as e:
        return f"Errore imprevisto: {e}"

# Esempio di utilizzo:
# contenuto = file_a_stringa("mio_testo.txt")
# print(contenuto)

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# --- 1. CONFIGURAZIONE ---
# Inserisci qui il nome esatto della cartella dove hai salvato il tuo modello migliore
# (es. "bert_medico_full" oppure "bert_medico_10_shot")
CARTELLA_MODELLO = "model/mul_bert_medico_full_shot" 
#MODEL_NAME = "dbmdz/bert-base-italian-cased"
MODEL_NAME = "bert-base-multilingual-cased"

print(f"Caricamento del modello da: {CARTELLA_MODELLO}...")

# Carichiamo il Tokenizer base e il TUO modello addestrato
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
modello = AutoModelForTokenClassification.from_pretrained(CARTELLA_MODELLO)

# Creiamo la "Pipeline" (è uno strumento di Hugging Face che fa tutto il lavoro sporco 
# di tokenizzazione, predizione e ri-allineamento delle parole)
# aggregation_strategy="simple" unisce in automatico i sub-token ("elettro" + "##cardio")
ner_pipeline = pipeline("token-classification", model=modello, tokenizer=tokenizer, aggregation_strategy="simple")

# --- 2. IL TEST DAL VIVO ---
print("\nScrivi un referto medico inventato (o premi Invio per usare l'esempio).")
print("Digita 'esci' per terminare.")

while True:
    testo_input = input("\nReferto: ")
    
    if testo_input.lower() == 'esci':
        break
        
    if not testo_input.strip():
        # Frase di esempio se premi solo Invio
        testo_input = file_a_stringa("example/fr_example.txt")
        print(f"Uso l'esempio: {testo_input}")

    # Chiediamo al modello di trovare le malattie!
    risultati = ner_pipeline(testo_input)
    
    print("\n--- RISULTATI ESTRATTI DA BERT FT ---")
    if not risultati:
        print("Nessuna entità clinica trovata.")
    else:
        for entita in risultati:
            parola = entita['word']
            etichetta = entita['entity_group']
            score = entita['score'] * 100
            
            # Stampiamo il risultato pulito
            print(f" Trovato: '{parola}'")
            print(f"   Tipo: {etichetta} (Score del modello: {score:.1f}%)\n")

Caricamento del modello da: model/mul_bert_medico_full_shot...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1139.87it/s, Materializing param=classifier.weight]                                     



Scrivi un referto medico inventato (o premi Invio per usare l'esempio).
Digita 'esci' per terminare.
Uso l'esempio: Rapport médical

Le patient, André Martin, âgé de 64 ans, consulte pour une fatigue intense et persistante, une douleur thoracique intermittente, des troubles respiratoires progressifs et des douleurs abdominales récurrentes. Il présente des antécédents d’hypertension artérielle, d’insuffisance cardiaque chronique, de diabète de type 2, d’insuffisance rénale modérée et d’arthrose lombaire. Depuis plusieurs semaines, il décrit une aggravation de l’essoufflement à l’effort, des épisodes de palpitations, un gonflement des membres inférieurs, ainsi qu’une prise de poids récente associée à une rétention hydrique. Il rapporte également une toux productive avec expectorations blanchâtres, des sifflements respiratoires nocturnes et une sensation d’oppression thoracique.

Sur le plan digestif, le patient mentionne des brûlures rétro-sternales, des douleurs épigastriques après les